# Home Credit: EDA и baseline
Файл: application_train.csv (скачать с Kaggle: home-credit-default-risk)
Положи данные в data/raw/ (папку в .gitignore!)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.features import build_application_features as build_features
from src.metrics import ks_stat, report
from src.models import make_logreg, cv_oof, RANDOM_STATE

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv("data/raw/application_train.csv")
print(df.shape)
df.head()

## 1. Общая картина

In [ ]:
# Баланс классов
print(df["TARGET"].value_counts())
print(f"Доля дефолтов: {df['TARGET'].mean():.2%}")

In [ ]:
# Типы признаков
print(df.dtypes.value_counts())

In [ ]:
# Пропуски: топ-20 колонок
missing = (df.isna().mean().sort_values(ascending=False) * 100).round(1)
print(missing.head(20))
print(f"\nКолонок с >50% пропусков: {(missing > 50).sum()} из {df.shape[1]}")

## 2. Аномалии

In [ ]:
# DAYS_EMPLOYED: заглушка 365243 (≈ 1000 лет)
print(df["DAYS_EMPLOYED"].describe())
anom = df["DAYS_EMPLOYED"] == 365243
print(f"\nЗаглушка: {anom.sum()} строк ({anom.mean():.1%})")
print(f"Дефолт у аномальных: {df.loc[anom, 'TARGET'].mean():.2%}")
print(f"Дефолт у остальных:  {df.loc[~anom, 'TARGET'].mean():.2%}")
print(df.loc[anom, "NAME_INCOME_TYPE"].value_counts())

In [ ]:
# Другие подозрительные места
print(df["CODE_GENDER"].value_counts())  # есть 'XNA'
print(df["OBS_30_CNT_SOCIAL_CIRCLE"].describe())  # выбросы (max ~348)
print(df["AMT_INCOME_TOTAL"].describe())  # есть экстремальный доход

## 3. Связь признаков с таргетом

In [ ]:
def default_rate_by(col, data=df, top=None, ax=None):
    """Доля дефолтов по категориям + размер группы."""
    g = data.groupby(col)["TARGET"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    if top:
        g = g.head(top)
    return g

for c in ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "OCCUPATION_TYPE", "CODE_GENDER"]:
    print(f"\n--- {c} ---")
    print(default_rate_by(c).round(3))

In [ ]:
# Возраст
df["AGE"] = -df["DAYS_BIRTH"] / 365
df["AGE_BIN"] = pd.cut(df["AGE"], bins=range(20, 75, 5))
df.groupby("AGE_BIN", observed=True)["TARGET"].mean().plot(kind="bar", figsize=(8, 4))
plt.title("Доля дефолтов по возрасту")
plt.ylabel("Default rate")
plt.show()

In [ ]:
# EXT_SOURCE: распределения для дефолтных и недефолтных
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]):
    sns.kdeplot(data=df, x=col, hue="TARGET", common_norm=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Корреляции числовых признаков с таргетом
corr = df.select_dtypes("number").corr()["TARGET"].drop("TARGET").sort_values()
print("Топ отрицательных:\n", corr.head(8).round(3))
print("\nТоп положительных:\n", corr.tail(8).round(3))

## 4. Подготовка признаков
Одна функция для всех преобразований: потом её же будет вызывать бот.

In [ ]:
# def build_features(data: pd.DataFrame, domain: bool = True) -> pd.DataFrame:
#     d = data.copy()
#
#     # аномалия в стаже -> NaN + флаг
#     d["DAYS_EMPLOYED_ANOM"] = (d["DAYS_EMPLOYED"] == 365243).astype(int)
#     d["DAYS_EMPLOYED"] = d["DAYS_EMPLOYED"].replace(365243, np.nan)
#
#     # 'XNA' в поле пола -> NaN
#     d["CODE_GENDER"] = d["CODE_GENDER"].replace("XNA", np.nan)
#
#     # дни -> годы (положительные)
#     d["AGE"] = -d["DAYS_BIRTH"] / 365
#
#     if domain:
#         d["CREDIT_INCOME_RATIO"] = d["AMT_CREDIT"] / d["AMT_INCOME_TOTAL"]
#         d["ANNUITY_INCOME_RATIO"] = d["AMT_ANNUITY"] / d["AMT_INCOME_TOTAL"]
#         d["CREDIT_TERM"] = d["AMT_ANNUITY"] / d["AMT_CREDIT"]
#         d["CREDIT_GOODS_RATIO"] = d["AMT_CREDIT"] / d["AMT_GOODS_PRICE"]
#         d["EMPLOYED_BIRTH_RATIO"] = d["DAYS_EMPLOYED"] / d["DAYS_BIRTH"]
#         d["INCOME_PER_PERSON"] = d["AMT_INCOME_TOTAL"] / d["CNT_FAM_MEMBERS"]
#
#         ext = d[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
#         d["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
#         d["EXT_SOURCE_MIN"] = ext.min(axis=1)
#         d["EXT_SOURCE_PROD"] = ext.prod(axis=1, skipna=False)
#
#     return d.replace([np.inf, -np.inf], np.nan)

## 5. Метрики и baseline

In [ ]:
# def ks_stat(y_true, y_score):
#     return ks_2samp(y_score[y_true == 1], y_score[y_true == 0]).statistic


# def report(name, y_true, y_score):
#     auc = roc_auc_score(y_true, y_score)
#     res = {
#         "model": name,
#         "ROC AUC": round(auc, 4),
#         "Gini": round(2 * auc - 1, 4),
#         "PR AUC": round(average_precision_score(y_true, y_score), 4),
#         "KS": round(ks_stat(y_true, y_score), 4),
#     }
#     return res


# def make_logreg(X: pd.DataFrame) -> Pipeline:
#     num_cols = X.select_dtypes("number").columns.tolist()
#     cat_cols = X.select_dtypes(exclude="number").columns.tolist()
#
#     pre = ColumnTransformer([
#         ("num", Pipeline([
#             ("imp", SimpleImputer(strategy="median", add_indicator=True)),
#             ("sc", StandardScaler()),
#         ]), num_cols),
#         ("cat", Pipeline([
#             ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
#             ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
#         ]), cat_cols),
#     ])
#     return Pipeline([
#         ("pre", pre),
#         ("clf", LogisticRegression(C=0.1, max_iter=2000)),
#     ])


# def cv_oof(model, X, y, n_splits=5):
#     """Out-of-fold предсказания: каждая строка предсказана моделью, которая её не видела."""
#     skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
#     return cross_val_predict(model, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]

In [ ]:
y = df["TARGET"]
drop_cols = ["TARGET", "SK_ID_CURR", "AGE_BIN"]  # AGE_BIN создан выше для графика

results = []

# (a) сырые признаки, только минимальная обработка
X_raw = df.drop(columns=drop_cols + ["AGE"])
results.append(report("LogReg: raw", y, cv_oof(make_logreg(X_raw), X_raw, y)))

# (b) + очистка аномалий
X_clean = build_features(df.drop(columns=["AGE_BIN", "AGE"]), domain=False).drop(columns=["TARGET", "SK_ID_CURR"])
results.append(report("LogReg: + cleaning", y, cv_oof(make_logreg(X_clean), X_clean, y)))

# (c) + доменные признаки
X_dom = build_features(df.drop(columns=["AGE_BIN", "AGE"]), domain=True).drop(columns=["TARGET", "SK_ID_CURR"])
results.append(report("LogReg: + domain features", y, cv_oof(make_logreg(X_dom), X_dom, y)))

pd.DataFrame(results)

Это первая ступенька «лесенки улучшений». Сохрани таблицу в README.
Дальше: агрегаты из bureau / previous_application, затем бустинг.